# Baseline LLM lengkap di Colab

Notebook ini menulis seluruh kode langsung di sel: model bahasa, pretraining, SFT, reward model, DPO, PPO, GRPO, evaluasi, dan generasi. Tidak ada impor dari repo lokal, sel writefile, atau langkah upload berkas.

**Data sengaja kosong.** Isi link dataset Hugging Face pada sel pengaturan. Setiap tahap tetap nonaktif sampai link dan sakelarnya diisi. Gunakan Runtime > Change runtime type > GPU untuk training. Checkpoint disimpan sementara di /content dan dapat diunduh pada sel terakhir.

Skema data yang diperlukan:

| Tahap | Kolom |
|---|---|
| Pretraining | text |
| SFT | prompt, response |
| Reward dan DPO | prompt, chosen, rejected |
| PPO dan GRPO | prompt; gold opsional untuk hadiah exact-match |

Alamat dapat berupa hf://datasets/owner/repo/path/file.parquet atau URL HTTPS ke Parquet, CSV, atau JSONL. Untuk dataset gated, terima syaratnya di situs Hugging Face dahulu dan aktifkan login di sel pengaturan. Contoh preferensi dan hadiah di notebook ini adalah algoritme; kualitasnya bergantung pada label yang Anda berikan. PPO dan GRPO adalah dua cabang dari model SFT, bukan tahap yang harus dijalankan berurutan.

### Urutan menjalankan

1. Buka notebook ini di Google Colab dan pilih GPU.
2. Jalankan sel dependensi dan pengaturan. Isi `DATA_URLS` dengan link dataset Anda; sesuaikan `COLUMNS` bila nama kolom berbeda. Jangan menaruh token akses dalam notebook yang akan dibagikan.
3. Jalankan semua sel definisi fungsi. Semua `RUN_*` bernilai `False`, sehingga belum ada training.
4. Aktifkan `RUN_PRETRAIN=True`, jalankan lagi sel pretraining, dan periksa metrik `base.pt`.
5. Aktifkan `RUN_SFT=True` setelah dataset SFT tersedia. Untuk preferensi, aktifkan `RUN_REWARD` dan/atau `RUN_DPO` sesuai tujuan.
6. PPO membutuhkan checkpoint SFT dan reward. GRPO membutuhkan SFT serta reward model atau kolom `gold` bila mode `exact`. Jalankan hanya cabang yang ingin diuji.
7. Pilih `SELECTED_STAGE`, aktifkan `RUN_GENERATE`, periksa beberapa prompt, lalu unduh ZIP bila perlu.

Jika memakai menu **Run all**, sel yang sakelarnya `False` akan dilewati. Saat sebuah sakelar diubah, jalankan lagi sel tahap tersebut; tidak perlu mengulang semua training. Semua data dan checkpoint di `/content` akan hilang ketika runtime Colab direset kecuali Anda mengunduhnya.

## 1. Dependensi dan pengaturan

Semua link dataset diawali string kosong sesuai permintaan. Isi link dan nama kolom yang cocok dengan dataset Anda. RUN_PRETRAIN dan sakelar tahap lain tetap False sampai Anda siap.

In [ ]:
%pip -q install -U pandas pyarrow fsspec huggingface_hub

In [ ]:
import json
import random
import re
import unicodedata
from pathlib import Path

import pandas as pd
import torch
from IPython.display import display
from torch import nn
from torch.nn import functional as F

DATA_URLS = {
    "pretrain": "",
    "sft": "",
    "preference": "",
    "rl_prompts": "",
}
COLUMNS = {
    "pretrain_text": "text",
    "sft_prompt": "prompt", "sft_response": "response",
    "pref_prompt": "prompt", "pref_chosen": "chosen", "pref_rejected": "rejected",
    "rl_prompt": "prompt", "rl_gold": "gold",
}
RUN_PRETRAIN = False
RUN_SFT = False
RUN_REWARD = False
RUN_DPO = False
RUN_PPO = False
RUN_GRPO = False
HF_GATED_LOGIN = False
SEED = 42
VALID_FRACTION = 0.20
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
OUTPUT_DIR = Path("/content/baseline_llm_runs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

MODEL_CONFIG = {"context_length": 512, "n_embed": 384, "n_head": 6, "n_blocks": 6}
PRETRAIN_CONFIG = {
    "steps": 800, "batch_size": 4, "grad_accum": 2,
    "eval_every": 100, "patience_evals": 5,
    "lr": 3e-4, "weight_decay": 0.01, "grad_clip": 1.0,
}
SFT_CONFIG = {"epochs": 2, "max_steps": 500, "eval_every": 50, "patience_evals": 4, "lr": 1e-4}
REWARD_CONFIG = {"epochs": 2, "max_steps": 300, "eval_every": 50, "lr": 1e-4}
DPO_CONFIG = {"epochs": 2, "max_steps": 300, "eval_every": 50, "beta": 0.1, "lr": 3e-5}
PPO_CONFIG = {
    "iterations": 20, "prompts_per_iter": 4, "response_bytes": 48,
    "update_epochs": 2, "clip": 0.2, "gamma": 1.0, "gae_lambda": 0.95,
    "kl_coef": 0.02, "value_coef": 0.5, "lr": 1e-5,
}
GRPO_CONFIG = {
    "iterations": 20, "prompts_per_iter": 2, "group_size": 4,
    "response_bytes": 48, "update_epochs": 2, "clip": 0.2,
    "kl_coef": 0.02, "lr": 1e-5, "reward_mode": "rm",
}

random.seed(SEED)
torch.manual_seed(SEED)
if DEVICE == "cuda":
    torch.cuda.manual_seed_all(SEED)
print("Device:", DEVICE, "| semua tahap:", "nonaktif" if not any(
    [RUN_PRETRAIN, RUN_SFT, RUN_REWARD, RUN_DPO, RUN_PPO, RUN_GRPO]
) else "sesuai sakelar")

## 2. Baca tabel dari link Hugging Face

Loader ini hanya menerima link hf:// atau HTTPS. Ia tidak membaca file lokal atau meminta upload. Kolom diperiksa sebelum training, sehingga kesalahan skema terlihat jelas. Pemisahan train/validasi terjadi per baris dokumen, sebelum token digabung.

In [ ]:
def maybe_login():
    if HF_GATED_LOGIN:
        from huggingface_hub import login
        login()

def load_remote_table(stage, required_columns):
    url = DATA_URLS[stage].strip()
    if not url:
        raise ValueError(f"Isi DATA_URLS[{stage!r}] terlebih dahulu.")
    if not url.startswith(("hf://", "https://")):
        raise ValueError("Gunakan URL hf:// atau https://; upload dan path lokal tidak dipakai.")
    suffix = url.split("?")[0].lower()
    maybe_login()
    if suffix.endswith(".parquet"):
        frame = pd.read_parquet(url)
    elif suffix.endswith((".jsonl", ".json")):
        frame = pd.read_json(url, lines=suffix.endswith(".jsonl"))
    elif suffix.endswith(".csv"):
        frame = pd.read_csv(url)
    else:
        raise ValueError("Format link harus .parquet, .jsonl, .json, atau .csv.")
    missing = [name for name in required_columns if name not in frame.columns]
    if missing:
        raise ValueError(f"Kolom hilang: {missing}. Tersedia: {list(frame.columns)}")
    print(stage, "baris:", len(frame), "kolom:", list(frame.columns))
    return frame

def clean_text(value):
    if isinstance(value, bytes):
        value = value.decode("utf-8", errors="replace")
    if not isinstance(value, str):
        return ""
    value = unicodedata.normalize("NFC", value)
    return re.sub(r"\s+", " ", value).strip()

def clean_rows(frame, columns):
    rows, seen = [], set()
    for values in frame[columns].itertuples(index=False, name=None):
        item = tuple(clean_text(value) for value in values)
        if any(not part for part in item):
            continue
        key = tuple(part.casefold() for part in item)
        if key not in seen:
            seen.add(key)
            rows.append(item)
    if len(rows) < 5:
        raise ValueError(f"Hanya {len(rows)} baris lengkap dan unik; perlu sedikitnya 5.")
    return rows

def split_rows(rows):
    shuffled = list(rows)
    random.Random(SEED).shuffle(shuffled)
    n_valid = max(1, min(len(shuffled) - 2, round(len(shuffled) * VALID_FRACTION)))
    return shuffled[n_valid:], shuffled[:n_valid]

def stage_rows(stage, keys):
    columns = [COLUMNS[key] for key in keys]
    frame = load_remote_table(stage, columns)
    rows = clean_rows(frame, columns)
    train, valid = split_rows(rows)
    print(f"{stage}: {len(train)} train, {len(valid)} valid")
    return train, valid

## 3. Tokenizer byte dan Transformer decoder

Kosakata 0 sampai 255 mewakili byte UTF-8; ID 256 menandai akhir dokumen. Ini berbeda dari tokenizer BPE pada pipeline lama, sehingga checkpoint lama tidak kompatibel. Model dan ukuran bawaan dibuat lebih kecil agar realistis untuk satu GPU Colab.

In [ ]:
EOS = 256
VOCAB_SIZE = 257

def encode(text, add_eos=False):
    ids = list(text.encode("utf-8"))
    return ids + [EOS] if add_eos else ids

def decode(ids):
    content = []
    for token in ids:
        if token == EOS:
            break
        if 0 <= token < 256:
            content.append(token)
    return bytes(content).decode("utf-8", errors="replace")

def make_stream(documents):
    tokens = []
    for text in documents:
        tokens.extend(encode(text, add_eos=True))
    return torch.tensor(tokens, dtype=torch.long)

class Attention(nn.Module):
    def __init__(self, width, heads):
        super().__init__()
        if width % heads:
            raise ValueError("n_embed harus habis dibagi n_head")
        self.heads = heads
        self.head_width = width // heads
        self.qkv = nn.Linear(width, 3 * width, bias=False)
        self.proj = nn.Linear(width, width)

    def forward(self, x):
        batch, length, width = x.shape
        q, k, v = self.qkv(x).split(width, dim=-1)
        q = q.reshape(batch, length, self.heads, self.head_width).transpose(1, 2)
        k = k.reshape(batch, length, self.heads, self.head_width).transpose(1, 2)
        v = v.reshape(batch, length, self.heads, self.head_width).transpose(1, 2)
        attended = F.scaled_dot_product_attention(q, k, v, is_causal=True)
        return self.proj(attended.transpose(1, 2).reshape(batch, length, width))

class MLP(nn.Module):
    def __init__(self, width):
        super().__init__()
        self.hidden = nn.Linear(width, 4 * width)
        self.proj = nn.Linear(4 * width, width)

    def forward(self, x):
        return self.proj(F.relu(self.hidden(x)))

class Block(nn.Module):
    def __init__(self, width, heads):
        super().__init__()
        self.ln_attention = nn.LayerNorm(width)
        self.attention = Attention(width, heads)
        self.ln_mlp = nn.LayerNorm(width)
        self.mlp = MLP(width)

    def forward(self, x):
        x = x + self.attention(self.ln_attention(x))
        return x + self.mlp(self.ln_mlp(x))

class CausalLM(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = dict(config)
        width = config["n_embed"]
        self.token_embedding = nn.Embedding(VOCAB_SIZE, width)
        self.position_embedding = nn.Embedding(config["context_length"], width)
        self.blocks = nn.ModuleList(
            Block(width, config["n_head"]) for _ in range(config["n_blocks"])
        )
        self.final_norm = nn.LayerNorm(width)
        self.lm_head = nn.Linear(width, VOCAB_SIZE, bias=False)
        self.lm_head.weight = self.token_embedding.weight

    def forward_hidden(self, tokens):
        batch, length = tokens.shape
        if length == 0 or length > self.config["context_length"]:
            raise ValueError("Panjang input harus berada di 1..context_length")
        positions = torch.arange(length, device=tokens.device)
        x = self.token_embedding(tokens) + self.position_embedding(positions)
        for block in self.blocks:
            x = block(x)
        return self.final_norm(x)

    def forward(self, tokens, targets=None):
        logits = self.lm_head(self.forward_hidden(tokens))
        loss = None
        if targets is not None:
            if logits.shape[:2] != targets.shape:
                raise ValueError(f"Input {logits.shape[:2]} dan target {targets.shape} tidak sejajar")
            loss = F.cross_entropy(logits.reshape(-1, VOCAB_SIZE), targets.reshape(-1))
        return logits, loss

def new_model():
    return CausalLM(MODEL_CONFIG).to(DEVICE)

## 4. Checkpoint, evaluasi, dan utilitas bersama

Checkpoint dibuat hanya ketika diminta tahap training. Tidak ada file sumber Python yang ditulis. Pretraining dan SFT menyimpan bobot dengan loss validasi terbaik; tahap lain menyimpan checkpoint tahapnya sendiri.

In [ ]:
def checkpoint_path(stage):
    return OUTPUT_DIR / f"{stage}.pt"

def save_stage(stage, model, metrics=None):
    path = checkpoint_path(stage)
    payload = {
        "stage": stage, "model_config": MODEL_CONFIG,
        "model": {key: value.detach().cpu() for key, value in model.state_dict().items()},
        "metrics": metrics or {},
    }
    torch.save(payload, path)
    return path

def load_stage(stage, model_factory=None):
    path = checkpoint_path(stage)
    if not path.exists():
        raise FileNotFoundError(f"Checkpoint {path} belum tersedia. Jalankan tahap sebelumnya.")
    payload = torch.load(path, map_location="cpu", weights_only=True)
    if payload["model_config"] != MODEL_CONFIG:
        raise ValueError("Konfigurasi model berbeda dari checkpoint; gunakan ukuran model yang sama.")
    model = model_factory() if model_factory is not None else new_model()
    model.load_state_dict(payload["model"], strict=True)
    return model.to(DEVICE)

def freeze(model):
    model.eval()
    for parameter in model.parameters():
        parameter.requires_grad_(False)
    return model

def write_metrics(stage, rows):
    path = OUTPUT_DIR / f"{stage}_metrics.jsonl"
    with path.open("w", encoding="utf-8") as handle:
        for row in rows:
            handle.write(json.dumps(row, ensure_ascii=False) + "\n")
    return path

def train_optimizer(model, lr):
    return torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)

def lm_batch(ids, batch_size, generator):
    context = MODEL_CONFIG["context_length"]
    if len(ids) <= context:
        raise ValueError("Aliran token train terlalu pendek; kurangi context_length atau tambah data.")
    starts = torch.randint(len(ids) - context, (batch_size,), generator=generator)
    positions = starts[:, None] + torch.arange(context)[None, :]
    return ids[positions].to(DEVICE), ids[positions + 1].to(DEVICE)

@torch.no_grad()
def full_lm_eval(model, ids):
    model.eval()
    context = MODEL_CONFIG["context_length"]
    if len(ids) < 2:
        raise ValueError("Validasi memerlukan sedikitnya dua token.")
    total_loss, correct, count = 0.0, 0, 0
    for start in range(0, len(ids) - 1, context):
        end = min(start + context, len(ids) - 1)
        x = ids[start:end].unsqueeze(0).to(DEVICE)
        y = ids[start + 1:end + 1].unsqueeze(0).to(DEVICE)
        logits, loss = model(x, y)
        size = y.numel()
        total_loss += loss.item() * size
        correct += (logits.argmax(-1) == y).sum().item()
        count += size
    return {"valid_loss": total_loss / count, "valid_byte_accuracy": correct / count}

## 5. Pretraining: menebak byte berikutnya

Model belajar dari teks mentah. Loss adalah cross entropy token berikutnya. Validasi menghitung seluruh pasangan token, termasuk blok terakhir yang lebih pendek; panjang input dan target selalu sama. `base.pt` berisi checkpoint dengan loss validasi terendah. Akurasi byte berguna untuk memantau proses, tetapi belum mengukur kualitas percakapan.

In [ ]:
def train_pretrain():
    train_rows, valid_rows = stage_rows("pretrain", ["pretrain_text"])
    train_ids = make_stream([row[0] for row in train_rows])
    valid_ids = make_stream([row[0] for row in valid_rows])
    model = new_model()
    optimizer = train_optimizer(model, PRETRAIN_CONFIG["lr"])
    generator = torch.Generator().manual_seed(SEED)
    best, stale, metrics = float("inf"), 0, []

    for step in range(PRETRAIN_CONFIG["steps"] + 1):
        if step == 0 or step % PRETRAIN_CONFIG["eval_every"] == 0 or step == PRETRAIN_CONFIG["steps"]:
            result = full_lm_eval(model, valid_ids)
            record = {"step": step, **result}
            metrics.append(record)
            print("pretrain", record)
            if result["valid_loss"] < best:
                best, stale = result["valid_loss"], 0
                save_stage("base", model, record)
            else:
                stale += 1
                if stale >= PRETRAIN_CONFIG["patience_evals"]:
                    print("Early stopping pada step", step)
                    break
        if step == PRETRAIN_CONFIG["steps"]:
            break
        model.train()
        optimizer.zero_grad(set_to_none=True)
        losses = []
        for _ in range(PRETRAIN_CONFIG["grad_accum"]):
            x, y = lm_batch(train_ids, PRETRAIN_CONFIG["batch_size"], generator)
            _, loss = model(x, y)
            (loss / PRETRAIN_CONFIG["grad_accum"]).backward()
            losses.append(loss.item())
        nn.utils.clip_grad_norm_(model.parameters(), PRETRAIN_CONFIG["grad_clip"])
        optimizer.step()
        if (step + 1) % PRETRAIN_CONFIG["eval_every"] == 0:
            print("train_loss", sum(losses) / len(losses))
    write_metrics("pretrain", metrics)
    return pd.DataFrame(metrics)

if RUN_PRETRAIN:
    pretrain_history = train_pretrain()
    display(pretrain_history.tail())
else:
    print("Pretraining dilewati. Isi DATA_URLS['pretrain'] dan ubah RUN_PRETRAIN=True bila siap.")

## 6. SFT: belajar menjawab contoh prompt

SFT melanjutkan `base.pt`. Prompt menjadi konteks; loss hanya dihitung pada token jawaban dan EOS. Dengan begitu model tidak diberi hadiah karena sekadar menyalin prompt. Kolom response harus berisi jawaban yang memang diinginkan.

In [ ]:
def response_tensors(prompt, response):
    context = MODEL_CONFIG["context_length"]
    prompt_ids = encode(prompt)[-max(1, context // 2):]
    response_ids = encode(response) + [EOS]
    response_ids = response_ids[:context + 1 - len(prompt_ids)]
    if not prompt_ids or not response_ids:
        raise ValueError("Prompt dan respons tidak boleh kosong.")
    joined = prompt_ids + response_ids
    x = torch.tensor(joined[:-1], dtype=torch.long, device=DEVICE).unsqueeze(0)
    y = torch.tensor(joined[1:], dtype=torch.long, device=DEVICE).unsqueeze(0)
    first_response_position = len(prompt_ids) - 1
    return x, y, first_response_position

def response_logprobs(model, prompt, response):
    x, y, first = response_tensors(prompt, response)
    logits, _ = model(x)
    logprobs = F.log_softmax(logits[:, first:], dim=-1)
    return logprobs.gather(-1, y[:, first:].unsqueeze(-1)).squeeze(-1).squeeze(0)

@torch.no_grad()
def sft_eval(model, rows):
    model.eval()
    total, count = 0.0, 0
    for prompt, response in rows:
        logprobs = response_logprobs(model, prompt, response)
        total -= logprobs.sum().item()
        count += logprobs.numel()
    return total / count

def train_sft():
    train_rows, valid_rows = stage_rows("sft", ["sft_prompt", "sft_response"])
    model = load_stage("base")
    optimizer = train_optimizer(model, SFT_CONFIG["lr"])
    best, stale, step, metrics = float("inf"), 0, 0, []
    order = random.Random(SEED)
    for epoch in range(SFT_CONFIG["epochs"]):
        order.shuffle(train_rows)
        for prompt, response in train_rows:
            model.train()
            optimizer.zero_grad(set_to_none=True)
            loss = -response_logprobs(model, prompt, response).mean()
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            step += 1
            if step % SFT_CONFIG["eval_every"] == 0 or step == SFT_CONFIG["max_steps"]:
                valid_loss = sft_eval(model, valid_rows)
                record = {"step": step, "train_loss": loss.item(), "valid_loss": valid_loss}
                metrics.append(record)
                print("sft", record)
                if valid_loss < best:
                    best, stale = valid_loss, 0
                    save_stage("sft", model, record)
                else:
                    stale += 1
                if stale >= SFT_CONFIG["patience_evals"]:
                    break
            if step >= SFT_CONFIG["max_steps"]:
                break
        if step >= SFT_CONFIG["max_steps"] or stale >= SFT_CONFIG["patience_evals"]:
            break
    if not metrics:
        record = {"step": step, "valid_loss": sft_eval(model, valid_rows)}
        metrics.append(record)
        save_stage("sft", model, record)
    write_metrics("sft", metrics)
    return pd.DataFrame(metrics)

if RUN_SFT:
    sft_history = train_sft()
    display(sft_history.tail())
else:
    print("SFT dilewati. Isi URL SFT, siapkan base.pt, lalu ubah RUN_SFT=True.")

## 7. Reward model: memilih jawaban yang lebih disukai

Dataset preferensi memuat prompt yang sama serta jawaban `chosen` dan `rejected`. Skor tiap jawaban berasal dari hidden state terakhir. Loss Bradley–Terry mendorong skor chosen melebihi rejected. Angka reward adalah skor relatif dalam dataset, bukan nilai kebenaran universal.

In [ ]:
class RewardModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.backbone = new_model()
        self.reward_head = nn.Linear(MODEL_CONFIG["n_embed"], 1)

    def forward(self, prompt, response):
        ids = encode(prompt + response, add_eos=True)[-MODEL_CONFIG["context_length"]:]
        tokens = torch.tensor(ids, dtype=torch.long, device=DEVICE).unsqueeze(0)
        hidden = self.backbone.forward_hidden(tokens)
        return self.reward_head(hidden[:, -1]).squeeze()

def new_reward_from_sft():
    model = RewardModel().to(DEVICE)
    model.backbone.load_state_dict(load_stage("sft").state_dict())
    return model

@torch.no_grad()
def reward_eval(model, rows):
    model.eval()
    losses, correct = [], 0
    for prompt, chosen, rejected in rows:
        difference = model(prompt, chosen) - model(prompt, rejected)
        losses.append(-F.logsigmoid(difference).item())
        correct += int(difference.item() > 0)
    return {"valid_loss": sum(losses) / len(losses), "pair_accuracy": correct / len(rows)}

def train_reward():
    train_rows, valid_rows = stage_rows(
        "preference", ["pref_prompt", "pref_chosen", "pref_rejected"]
    )
    train_rows = [row for row in train_rows if row[1] != row[2]]
    valid_rows = [row for row in valid_rows if row[1] != row[2]]
    if not train_rows or not valid_rows:
        raise ValueError("Perlu pasangan chosen/rejected yang berbeda pada train dan validasi.")
    model = new_reward_from_sft()
    optimizer = train_optimizer(model, REWARD_CONFIG["lr"])
    best, step, metrics = float("inf"), 0, []
    order = random.Random(SEED)
    for epoch in range(REWARD_CONFIG["epochs"]):
        order.shuffle(train_rows)
        for prompt, chosen, rejected in train_rows:
            model.train()
            optimizer.zero_grad(set_to_none=True)
            difference = model(prompt, chosen) - model(prompt, rejected)
            loss = -F.logsigmoid(difference)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            step += 1
            if step % REWARD_CONFIG["eval_every"] == 0 or step == REWARD_CONFIG["max_steps"]:
                record = {"step": step, "train_loss": loss.item(), **reward_eval(model, valid_rows)}
                metrics.append(record)
                print("reward", record)
                if record["valid_loss"] < best:
                    best = record["valid_loss"]
                    save_stage("reward", model, record)
            if step >= REWARD_CONFIG["max_steps"]:
                break
        if step >= REWARD_CONFIG["max_steps"]:
            break
    if not metrics:
        record = {"step": step, **reward_eval(model, valid_rows)}
        metrics.append(record)
        save_stage("reward", model, record)
    write_metrics("reward", metrics)
    return pd.DataFrame(metrics)

if RUN_REWARD:
    reward_history = train_reward()
    display(reward_history.tail())
else:
    print("Reward model dilewati. Isi URL preferensi dan jalankan SFT dahulu.")

## 8. DPO: melatih pilihan langsung

DPO memakai dataset preferensi yang sama, tetapi memperbarui model bahasa langsung. SFT dijadikan referensi beku. Loss membandingkan rasio probabilitas chosen/rejected antara model yang dilatih dan referensi.

In [ ]:
@torch.no_grad()
def dpo_eval(policy, reference, rows, beta):
    policy.eval()
    losses, correct = [], 0
    for prompt, chosen, rejected in rows:
        pc = response_logprobs(policy, prompt, chosen).sum()
        pr = response_logprobs(policy, prompt, rejected).sum()
        rc = response_logprobs(reference, prompt, chosen).sum()
        rr = response_logprobs(reference, prompt, rejected).sum()
        margin = beta * ((pc - pr) - (rc - rr))
        losses.append(-F.logsigmoid(margin).item())
        correct += int(margin.item() > 0)
    return {"valid_loss": sum(losses) / len(losses), "implicit_pair_accuracy": correct / len(rows)}

def train_dpo():
    train_rows, valid_rows = stage_rows(
        "preference", ["pref_prompt", "pref_chosen", "pref_rejected"]
    )
    train_rows = [row for row in train_rows if row[1] != row[2]]
    valid_rows = [row for row in valid_rows if row[1] != row[2]]
    if not train_rows or not valid_rows:
        raise ValueError("Perlu pasangan chosen/rejected yang berbeda pada train dan validasi.")
    policy = load_stage("sft")
    reference = freeze(load_stage("sft"))
    optimizer = train_optimizer(policy, DPO_CONFIG["lr"])
    beta = DPO_CONFIG["beta"]
    best, step, metrics = float("inf"), 0, []
    order = random.Random(SEED)
    for epoch in range(DPO_CONFIG["epochs"]):
        order.shuffle(train_rows)
        for prompt, chosen, rejected in train_rows:
            policy.train()
            optimizer.zero_grad(set_to_none=True)
            pc = response_logprobs(policy, prompt, chosen).sum()
            pr = response_logprobs(policy, prompt, rejected).sum()
            with torch.no_grad():
                rc = response_logprobs(reference, prompt, chosen).sum()
                rr = response_logprobs(reference, prompt, rejected).sum()
            loss = -F.logsigmoid(beta * ((pc - pr) - (rc - rr)))
            loss.backward()
            nn.utils.clip_grad_norm_(policy.parameters(), 1.0)
            optimizer.step()
            step += 1
            if step % DPO_CONFIG["eval_every"] == 0 or step == DPO_CONFIG["max_steps"]:
                record = {"step": step, "train_loss": loss.item(), **dpo_eval(policy, reference, valid_rows, beta)}
                metrics.append(record)
                print("dpo", record)
                if record["valid_loss"] < best:
                    best = record["valid_loss"]
                    save_stage("dpo", policy, record)
            if step >= DPO_CONFIG["max_steps"]:
                break
        if step >= DPO_CONFIG["max_steps"]:
            break
    if not metrics:
        record = {"step": step, **dpo_eval(policy, reference, valid_rows, beta)}
        metrics.append(record)
        save_stage("dpo", policy, record)
    write_metrics("dpo", metrics)
    return pd.DataFrame(metrics)

if RUN_DPO:
    dpo_history = train_dpo()
    display(dpo_history.tail())
else:
    print("DPO dilewati. Isi URL preferensi dan jalankan SFT dahulu.")

## 9. Alat bersama untuk PPO dan GRPO

Kedua metode mengambil sampel respons dari model SFT. PPO melatih policy dan value head dengan hadiah dari reward model. GRPO membandingkan beberapa jawaban untuk prompt yang sama; hadiahnya dapat berasal dari reward model atau exact match terhadap kolom `gold`. Token respons dibatasi agar semua operasi muat dalam context model.

In [ ]:
def rl_prompt_ids(prompt):
    ids = encode(prompt)[-max(1, MODEL_CONFIG["context_length"] // 2):]
    if not ids:
        raise ValueError("Prompt RL kosong.")
    return ids

def token_tensors(prompt_ids, response_ids):
    joined = prompt_ids + response_ids
    if len(joined) < 2 or len(joined) - 1 > MODEL_CONFIG["context_length"]:
        raise ValueError("Urutan RL melebihi context_length.")
    x = torch.tensor(joined[:-1], dtype=torch.long, device=DEVICE).unsqueeze(0)
    y = torch.tensor(response_ids, dtype=torch.long, device=DEVICE).unsqueeze(0)
    first = len(prompt_ids) - 1
    return x, y, first

def action_logprobs(model, prompt_ids, response_ids):
    x, y, first = token_tensors(prompt_ids, response_ids)
    logits, _ = model(x)
    selected = F.log_softmax(logits[:, first:], dim=-1).gather(-1, y.unsqueeze(-1))
    return selected.squeeze(-1).squeeze(0)

@torch.no_grad()
def sample_ids(model, prompt_ids, limit, temperature=1.0):
    model.eval()
    response = []
    max_tokens = min(limit, MODEL_CONFIG["context_length"] + 1 - len(prompt_ids))
    for _ in range(max_tokens):
        context = torch.tensor(
            (prompt_ids + response)[-MODEL_CONFIG["context_length"]:],
            dtype=torch.long, device=DEVICE
        ).unsqueeze(0)
        logits, _ = model(context)
        probs = F.softmax(logits[0, -1].float() / temperature, dim=-1)
        token = int(torch.multinomial(probs, 1).item())
        response.append(token)
        if token == EOS:
            break
    return response

def reward_for_response(reward_model, prompt, response_ids):
    with torch.no_grad():
        return torch.tanh(reward_model(prompt, decode(response_ids))).item()

class ActorCritic(nn.Module):
    def __init__(self):
        super().__init__()
        self.policy = new_model()
        self.value_head = nn.Linear(MODEL_CONFIG["n_embed"], 1)

    def forward(self, tokens):
        hidden = self.policy.forward_hidden(tokens)
        return self.policy.lm_head(hidden), self.value_head(hidden).squeeze(-1)

def actor_stats(actor, prompt_ids, response_ids):
    x, y, first = token_tensors(prompt_ids, response_ids)
    logits, values = actor(x)
    selected = F.log_softmax(logits[:, first:], dim=-1).gather(-1, y.unsqueeze(-1))
    return selected.squeeze(-1).squeeze(0), values[:, first:].squeeze(0)

def gae_advantages(rewards, old_values, gamma, lam):
    values = old_values.tolist()
    advantage, running = [], 0.0
    for index in range(len(rewards) - 1, -1, -1):
        next_value = values[index + 1] if index + 1 < len(values) else 0.0
        delta = rewards[index] + gamma * next_value - values[index]
        running = delta + gamma * lam * running
        advantage.append(running)
    advantage.reverse()
    advantages = torch.tensor(advantage, dtype=torch.float32, device=DEVICE)
    returns = advantages + old_values
    return advantages, returns

## 10. PPO: optimasi policy dengan value head

Reward akhir diberikan pada token terakhir. Selama rollout, setiap token juga menerima penalti KL terhadap model SFT yang beku. GAE menghitung advantage; PPO membatasi perubahan rasio probabilitas dengan clipping. Implementasi ini bertujuan belajar mekanismenya, dengan batch kecil dan tanpa infrastruktur RLHF skala produksi.

In [ ]:
def train_ppo():
    train_rows, valid_rows = stage_rows("rl_prompts", ["rl_prompt"])
    prompts = [row[0] for row in train_rows]
    actor = ActorCritic().to(DEVICE)
    actor.policy.load_state_dict(load_stage("sft").state_dict())
    reference = freeze(load_stage("sft"))
    reward_model = freeze(load_stage("reward", RewardModel))
    optimizer = train_optimizer(actor, PPO_CONFIG["lr"])
    rng = random.Random(SEED)
    metrics = []

    for iteration in range(1, PPO_CONFIG["iterations"] + 1):
        batch_prompts = rng.choices(prompts, k=PPO_CONFIG["prompts_per_iter"])
        trajectories = []
        for prompt in batch_prompts:
            pids = rl_prompt_ids(prompt)
            response = sample_ids(actor.policy, pids, PPO_CONFIG["response_bytes"])
            with torch.no_grad():
                old_logps, old_values = actor_stats(actor, pids, response)
                ref_logps = action_logprobs(reference, pids, response)
            score = reward_for_response(reward_model, prompt, response)
            rewards = (-PPO_CONFIG["kl_coef"] * (old_logps - ref_logps)).tolist()
            rewards[-1] += score
            advantages, returns = gae_advantages(
                rewards, old_values, PPO_CONFIG["gamma"], PPO_CONFIG["gae_lambda"]
            )
            trajectories.append((pids, response, old_logps, old_values, advantages, returns, score))

        all_advantages = torch.cat([item[4] for item in trajectories])
        center, scale = all_advantages.mean(), all_advantages.std(unbiased=False).clamp_min(1e-6)
        for _ in range(PPO_CONFIG["update_epochs"]):
            for pids, response, old_logps, old_values, advantages, returns, score in trajectories:
                actor.train()
                optimizer.zero_grad(set_to_none=True)
                logps, values = actor_stats(actor, pids, response)
                ratio = torch.exp(logps - old_logps)
                normalized = (advantages - center) / scale
                clipped = ratio.clamp(1 - PPO_CONFIG["clip"], 1 + PPO_CONFIG["clip"])
                policy_loss = -torch.minimum(ratio * normalized, clipped * normalized).mean()
                clipped_values = old_values + (values - old_values).clamp(
                    -PPO_CONFIG["clip"], PPO_CONFIG["clip"]
                )
                value_loss = 0.5 * torch.maximum(
                    (values - returns).square(), (clipped_values - returns).square()
                ).mean()
                loss = policy_loss + PPO_CONFIG["value_coef"] * value_loss
                loss.backward()
                nn.utils.clip_grad_norm_(actor.parameters(), 1.0)
                optimizer.step()
        record = {
            "iteration": iteration, "mean_reward": sum(item[6] for item in trajectories) / len(trajectories),
            "policy_loss": policy_loss.item(), "value_loss": value_loss.item(),
        }
        metrics.append(record)
        print("ppo", record)
        save_stage("ppo", actor, record)
    write_metrics("ppo", metrics)
    return pd.DataFrame(metrics)

if RUN_PPO:
    ppo_history = train_ppo()
    display(ppo_history.tail())
else:
    print("PPO dilewati. Perlu SFT, reward model, URL prompt RL, dan RUN_PPO=True.")

## 11. GRPO: membandingkan satu grup jawaban

Untuk setiap prompt, model membuat beberapa respons. Reward dinormalisasi di dalam grup; bila semuanya sama, tidak ada sinyal pembelajaran. Mode `rm` memakai reward model. Mode `exact` memerlukan kolom gold pada dataset RL dan memberi nilai 1 hanya saat respons cocok setelah spasi dirapikan dan huruf dikecilkan.

In [ ]:
def rl_examples_for_grpo():
    mode = GRPO_CONFIG["reward_mode"]
    if mode == "rm":
        train_rows, valid_rows = stage_rows("rl_prompts", ["rl_prompt"])
        return [(row[0], None) for row in train_rows]
    if mode == "exact":
        train_rows, valid_rows = stage_rows("rl_prompts", ["rl_prompt", "rl_gold"])
        return train_rows
    raise ValueError("GRPO reward_mode harus 'rm' atau 'exact'.")

def exact_reward(response_ids, gold):
    return float(clean_text(decode(response_ids)).casefold() == clean_text(gold).casefold())

def train_grpo():
    examples = rl_examples_for_grpo()
    policy = load_stage("sft")
    reference = freeze(load_stage("sft"))
    reward_model = (
        freeze(load_stage("reward", RewardModel))
        if GRPO_CONFIG["reward_mode"] == "rm" else None
    )
    optimizer = train_optimizer(policy, GRPO_CONFIG["lr"])
    rng = random.Random(SEED)
    metrics = []

    for iteration in range(1, GRPO_CONFIG["iterations"] + 1):
        trajectories, rewards_seen = [], []
        for prompt, gold in rng.choices(examples, k=GRPO_CONFIG["prompts_per_iter"]):
            pids = rl_prompt_ids(prompt)
            group = []
            for _ in range(GRPO_CONFIG["group_size"]):
                response = sample_ids(policy, pids, GRPO_CONFIG["response_bytes"])
                with torch.no_grad():
                    old_logps = action_logprobs(policy, pids, response)
                    ref_logps = action_logprobs(reference, pids, response)
                score = (
                    reward_for_response(reward_model, prompt, response)
                    if reward_model is not None else exact_reward(response, gold)
                )
                group.append((response, old_logps, ref_logps, score))
                rewards_seen.append(score)
            values = torch.tensor([item[3] for item in group], device=DEVICE)
            advantages = (values - values.mean()) / values.std(unbiased=False).clamp_min(1e-6)
            for (response, old_logps, ref_logps, score), advantage in zip(group, advantages):
                trajectories.append((pids, response, old_logps, ref_logps, advantage.detach()))

        for _ in range(GRPO_CONFIG["update_epochs"]):
            for pids, response, old_logps, ref_logps, advantage in trajectories:
                policy.train()
                optimizer.zero_grad(set_to_none=True)
                logps = action_logprobs(policy, pids, response)
                ratio = torch.exp(logps - old_logps)
                clipped = ratio.clamp(1 - GRPO_CONFIG["clip"], 1 + GRPO_CONFIG["clip"])
                surrogate = torch.minimum(ratio * advantage, clipped * advantage).mean()
                log_ratio = ref_logps - logps
                kl = (torch.exp(log_ratio) - log_ratio - 1).mean()
                loss = -surrogate + GRPO_CONFIG["kl_coef"] * kl
                loss.backward()
                nn.utils.clip_grad_norm_(policy.parameters(), 1.0)
                optimizer.step()
        record = {
            "iteration": iteration, "mean_reward": sum(rewards_seen) / len(rewards_seen),
            "reward_std": float(torch.tensor(rewards_seen).std(unbiased=False).item()),
            "loss": loss.item(),
        }
        metrics.append(record)
        print("grpo", record)
        save_stage("grpo", policy, record)
    write_metrics("grpo", metrics)
    return pd.DataFrame(metrics)

if RUN_GRPO:
    grpo_history = train_grpo()
    display(grpo_history.tail())
else:
    print("GRPO dilewati. Perlu SFT, URL prompt RL, reward model atau gold, dan RUN_GRPO=True.")

## 12. Bandingkan hasil dan buat teks

Loss pretraining/SFT dan pair accuracy reward/DPO mengukur hal berbeda, sehingga jangan diurutkan menjadi satu angka pemenang. Selain metrik, uji beberapa prompt yang tidak ada di data train dan baca jawabannya secara manual. Generasi di bawah membatasi byte agar UTF-8 tetap valid dan hasil tidak menjadi karakter pengganti `�`.

In [ ]:
def show_saved_metrics():
    rows = []
    for stage in ["base", "sft", "reward", "dpo", "ppo", "grpo"]:
        path = checkpoint_path(stage)
        if path.exists():
            payload = torch.load(path, map_location="cpu", weights_only=True)
            rows.append({"stage": stage, **payload.get("metrics", {})})
    if rows:
        display(pd.DataFrame(rows))
    else:
        print("Belum ada checkpoint. Jalankan tahap yang Anda pilih setelah mengisi dataset.")

show_saved_metrics()

In [ ]:
def utf8_state(prefix):
    try:
        prefix.decode("utf-8")
        return 0
    except UnicodeDecodeError as error:
        if error.reason == "unexpected end of data" and error.end == len(prefix):
            return len(prefix) - error.start
        return -1

@torch.no_grad()
def generate_text(model, prompt, max_new_bytes=160, temperature=0.8, top_k=40):
    if temperature <= 0:
        raise ValueError("temperature harus > 0")
    model.eval()
    prompt_ids = encode(prompt)
    if not prompt_ids:
        raise ValueError("prompt tidak boleh kosong")
    generated = bytearray()
    for _ in range(max_new_bytes):
        context = (prompt_ids + list(generated))[-MODEL_CONFIG["context_length"]:]
        x = torch.tensor(context, dtype=torch.long, device=DEVICE).unsqueeze(0)
        logits, _ = model(x)
        scores = logits[0, -1].float() / temperature
        allowed = torch.full_like(scores, float("-inf"))
        for token in range(VOCAB_SIZE):
            if token == EOS:
                if utf8_state(generated) == 0:
                    allowed[token] = scores[token]
            elif utf8_state(generated + bytes([token])) >= 0:
                allowed[token] = scores[token]
        if top_k is not None:
            k = min(top_k, VOCAB_SIZE)
            cutoff = torch.topk(allowed, k).values[-1]
            allowed = allowed.masked_fill(allowed < cutoff, float("-inf"))
        token = int(torch.multinomial(F.softmax(allowed, dim=-1), 1).item())
        if token == EOS:
            break
        generated.append(token)
    while utf8_state(generated) != 0:
        generated.pop()
    return prompt + generated.decode("utf-8")

SELECTED_STAGE = "sft"  # "base", "sft", "dpo", "ppo", atau "grpo"
PROMPT = "Pagi itu, "
RUN_GENERATE = False
if RUN_GENERATE:
    if SELECTED_STAGE == "ppo":
        model = load_stage("ppo", ActorCritic).policy
    elif SELECTED_STAGE in {"base", "sft", "dpo", "grpo"}:
        model = load_stage(SELECTED_STAGE)
    else:
        raise ValueError("SELECTED_STAGE tidak dikenal")
    print(generate_text(model, PROMPT))
else:
    print("Generasi dilewati. Pilih checkpoint dan ubah RUN_GENERATE=True bila siap.")

In [ ]:
EVALUATION_PROMPTS = [
    "Pagi itu, ",
    "Apa yang terjadi setelah hujan reda? ",
    "Ceritakan sebuah penemuan sederhana: ",
]
RUN_QUALITATIVE_EVAL = False
if RUN_QUALITATIVE_EVAL:
    stages = [stage for stage in ["base", "sft", "dpo", "ppo", "grpo"] if checkpoint_path(stage).exists()]
    comparisons = []
    for stage in stages:
        model = load_stage(stage, ActorCritic).policy if stage == "ppo" else load_stage(stage)
        for prompt in EVALUATION_PROMPTS:
            comparisons.append({"stage": stage, "prompt": prompt, "response": generate_text(model, prompt)})
    display(pd.DataFrame(comparisons))
else:
    print("Evaluasi kualitatif dilewati. Ubah RUN_QUALITATIVE_EVAL=True setelah ada checkpoint.")

## 13. Simpan hasil dari Colab

File checkpoint dan metrik berada di `/content/baseline_llm_runs`. Unduh ZIP bila ingin menyimpan hasil sebelum runtime Colab berakhir. Tahap ini juga nonaktif secara default.

In [ ]:
RUN_DOWNLOAD = False
if RUN_DOWNLOAD:
    import shutil
    from google.colab import files
    archive = shutil.make_archive("/content/baseline_llm_runs", "zip", OUTPUT_DIR)
    files.download(archive)
else:
    print("Unduhan dilewati. Ubah RUN_DOWNLOAD=True setelah training selesai.")